In [0]:
%spark.pyspark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [1]:
%spark.pyspark
users = spark.createDataFrame([
    ("u1", "Berlin"),
    ("u2", "Berlin"),
    ("u3", "Munich"),
    ("u4", "Hamburg")
], ["user_id", "city"])

orders = spark.createDataFrame([
    ("o1", "u1", "p1", 2, 10.0),
    ("o2", "u1", "p2", 1, 30.0),
    ("o3", "u2", "p1", 1, 10.0),
    ("o4", "u2", "p3", 5, 7.0),
    ("o5", "u3", "p2", 3, 30.0),
    ("o6", "u3", "p3", 1, 7.0),
    ("o7", "u4", "p1", 10, 10.0)
], ["order_id", "user_id", "product_id", "qty", "price"])

products = spark.createDataFrame([
    ("p1", "Ring VOLA"),
    ("p2", "Ring POROG"),
    ("p3", "Ring TISHINA")
], ["product_id", "product_name"])

users.show()
orders.show()
products.show()

In [2]:
%spark.pyspark
orders_with_revenue = orders.withColumn("revenue", F.col("qty") * F.col("price"))
orders_with_city = orders_with_revenue.join(users, on="user_id", how="inner")
full_data = orders_with_city.join(products, on="product_id", how="inner")
full_data.show()

In [3]:
%spark.pyspark
aggregated = full_data.groupBy("city", "product_id", "product_name").agg(
    F.count("order_id").alias("orders_cnt"),
    F.sum("qty").alias("qty_sum"),
    F.sum("revenue").alias("revenue_sum")
)
aggregated.show()

In [4]:
%spark.pyspark
window_spec = Window.partitionBy("city").orderBy(F.col("revenue_sum").desc())
ranked = aggregated.withColumn("rn", F.row_number().over(window_spec))
top2 = ranked.filter(F.col("rn") <= 2).drop("rn")
top2.show()

In [5]:
%spark.pyspark
hdfs_path = "/tmp/sandbox_zeppelin/mart_city_top_products/"
top2.write.mode("overwrite").parquet(hdfs_path)
print(f"Data saved to HDFS: {hdfs_path}")

In [6]:
%spark.pyspark
read_df = spark.read.parquet(hdfs_path)
read_df.show()

In [7]:
%spark.pyspark
